In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from confidence.utils import ModelInputOutputWrapper
from its.search import InverseTransformationSearch
from search.parallel_gradient import ParallelGradientDescent
from utils.affine_transforms_old import AffineTransformation2D
from utils.sampling import BatchNegativeSampler

#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "mnist"

default_architecutre_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "coil100": "coil_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

architecture = default_architecutre_mapping[dataset]
budget = None
from utils.transforms.apply import grid_resample_border, grid_resample_reflection

from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info, path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
from utils.transforms.apply import grid_resample
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="sobol"
    ).to(device)

In [ ]:
import torch
import torch.nn.functional as F
import kornia

def augment_true(images, max_shift=1.0, max_angle=3.0, max_scale=0.04):
    B, C, H, W = images.shape
    device = images.device

    # random translations in pixels
    translations = torch.empty(B, 2, device=device).uniform_(-max_shift, max_shift)

    # random angles in radians
    angles = torch.empty(B, device=device).uniform_(-max_angle, max_angle)

    # random scales
    scales = 1.0 + torch.empty(B, device=device).uniform_(-max_scale, max_scale)
    scales = scales.unsqueeze(1).repeat(1, 2)

    # center of image
    center = torch.tensor([[W / 2, H / 2]], device=device).expand(B, -1)

    # get 3x3 affine matrices in pixel coordinates
    M_3x3 = kornia.geometry.transform.get_affine_matrix2d(translations, center, scales, angles)

    # normalize to PyTorch [-1,1] coordinates
    norm_mat = torch.tensor([
        [2.0/(W-1), 0, -1],
        [0, 2.0/(H-1), -1],
        [0, 0, 1]
    ], device=device).unsqueeze(0).repeat(B,1,1)

    norm_mat_inv = torch.tensor([
        [(W-1)/2.0, 0, (W-1)/2.0],
        [0, (H-1)/2.0, (H-1)/2.0],
        [0, 0, 1]
    ], device=device).unsqueeze(0).repeat(B,1,1)

    # properly transform 3x3 matrices
    M_norm_3x3 = torch.bmm(norm_mat, torch.bmm(M_3x3, norm_mat_inv))

    # slice to 2x3 for affine_grid
    M_norm_2x3 = M_norm_3x3[:, :2, :]

    # create grid and warp
    grid = F.affine_grid(M_norm_2x3, images.size(), align_corners=False)
    res = F.grid_sample(images, grid, mode='bilinear', padding_mode='zeros', align_corners=False)

    return res

import torch
import torch.nn.functional as F
import kornia as K
from typing import Tuple, Callable, List, Optional

Tensor = torch.Tensor

def _rand_params(shape, device, low, high):
    return torch.empty(shape, device=device).uniform_(low, high)

@torch.no_grad()
def random_gaussian_blur(x: Tensor,
                         p: float = 0.5,
                         ksize_choices: Tuple[int, ...] = (3, 5),
                         sigma_range: Tuple[float, float] = (0.1, 1.5)) -> Tensor:
    """
    Vectorized per-sample Gaussian blur.
    """
    if p <= 0:
        return x
    B, C, H, W = x.shape
    device = x.device
    apply_mask = torch.rand(B, device=device) < p
    if not apply_mask.any():
        return x
    # Choose kernel size per sample
    k_choices = torch.tensor(ksize_choices, device=device)
    k_idx = torch.randint(0, len(k_choices), (B,), device=device)
    k_sizes = k_choices[k_idx]

    # Sample sigmas per sample (Kornia expects (B,2))
    sigmas = _rand_params((B, 2), device, *sigma_range)
    # Build blurred batch
    out = x.clone()
    # Group by kernel size to avoid looping over samples
    for k in k_choices.unique():
        mask_k = (k_sizes == k) & apply_mask
        if not mask_k.any():
            continue
        x_sub = x[mask_k]
        sigma_sub = sigmas[mask_k]
        blur = K.filters.GaussianBlur2d((int(k.item()), int(k.item())), sigma_sub)(x_sub)
        out[mask_k] = blur
    return out

@torch.no_grad()
def random_unsharp_mask(x: Tensor,
                        p: float = 0.5,
                        ksize: int = 5,
                        sigma_range: Tuple[float, float] = (0.5, 1.5),
                        amount_range: Tuple[float, float] = (0.5, 1.5),
                        clamp: bool = True) -> Tensor:
    """
    Sharpen / deblur via unsharp masking: y = x + a * (x - blur(x)).
    """
    if p <= 0:
        return x
    B = x.shape[0]
    device = x.device
    apply_mask = torch.rand(B, device=device) < p
    if not apply_mask.any():
        return x
    sigmas = _rand_params((B, 2), device, *sigma_range)
    amounts = _rand_params((B, 1, 1, 1), device, *amount_range)
    blur_all = K.filters.GaussianBlur2d((ksize, ksize), sigmas)(x)
    sharpened = x + amounts * (x - blur_all)
    if clamp:
        sharpened = sharpened.clamp(0.0, 1.0)
    return torch.where(apply_mask.view(B, 1, 1, 1), sharpened, x)

@torch.no_grad()
def random_gaussian_noise(x: Tensor,
                          p: float = 0.5,
                          sigma_range: Tuple[float, float] = (0.001, 0.05),
                          clamp: bool = True) -> Tensor:
    if p <= 0:
        return x
    B = x.shape[0]
    device = x.device
    mask = torch.rand(B, device=device) < p
    if not mask.any():
        return x
    sigmas = _rand_params((B, 1, 1, 1), device, *sigma_range)
    noise = torch.randn_like(x) * sigmas
    out = x + noise
    if clamp:
        out = out.clamp(0.0, 1.0)
    return torch.where(mask.view(B,1,1,1), out, x)

@torch.no_grad()
def random_contrast(x: Tensor,
                    p: float = 0.5,
                    factor_range: Tuple[float, float] = (0.75, 1.25),
                    eps: float = 1e-8) -> Tensor:
    if p <= 0:
        return x
    B = x.shape[0]
    device = x.device
    mask = torch.rand(B, device=device) < p
    if not mask.any():
        return x
    factors = _rand_params((B,1,1,1), device, *factor_range)
    mean = x.mean(dim=(2,3), keepdim=True)
    out = (x - mean) * factors + mean
    out = torch.where(mask.view(B,1,1,1), out, x)
    return out.clamp(0.0, 1.0)

@torch.no_grad()
def random_gamma(x: Tensor,
                 p: float = 0.5,
                 gamma_range: Tuple[float, float] = (0.7, 1.5),
                 eps: float = 1e-8) -> Tensor:
    if p <= 0:
        return x
    B = x.shape[0]
    device = x.device
    mask = torch.rand(B, device=device) < p
    if not mask.any():
        return x
    gammas = _rand_params((B,1,1,1), device, *gamma_range)
    out = (x.clamp(min=eps)) ** gammas
    return torch.where(mask.view(B,1,1,1), out, x)

@torch.no_grad()
def random_cutout(x: Tensor,
                  p: float = 0.5,
                  scale_range: Tuple[float, float] = (0.1, 0.25),
                  fill: float = 0.0) -> Tensor:
    """
    Rectangular mask with random size per sample.
    """
    if p <= 0:
        return x
    B, C, H, W = x.shape
    device = x.device
    mask_apply = torch.rand(B, device=device) < p
    if not mask_apply.any():
        return x
    out = x.clone()
    for_mask = torch.nonzero(mask_apply).flatten()
    # Vectorizable approach: build per-pixel mask
    yy = torch.arange(H, device=device).view(1, H, 1)
    xx = torch.arange(W, device=device).view(1, 1, W)
    for b in for_mask:
        scale = _rand_params((1,), device, *scale_range).item()
        cut_h = max(1, int(H * scale))
        cut_w = max(1, int(W * scale))
        cy = torch.randint(0, H, (1,), device=device).item()
        cx = torch.randint(0, W, (1,), device=device).item()
        y1 = max(0, cy - cut_h // 2); y2 = min(H, y1 + cut_h)
        x1 = max(0, cx - cut_w // 2); x2 = min(W, x1 + cut_w)
        out[b,:,y1:y2,x1:x2] = fill
    return out

@torch.no_grad()
def random_blur_or_sharpen(
    x: torch.Tensor,
    p: float = 0.4,
    prob_blur: float = 0.5,           # fraction of affected samples that get blur; rest sharpen
    blur_ks_choices=(3,5),
    blur_sigma_range=(0.2,1.2),
    usm_ksize=5,
    usm_sigma_range=(0.5,1.5),
    usm_amount_range=(0.5,1.3),
    clamp: bool = True
) -> torch.Tensor:
    """
    Per sample choose: no-op, Gaussian blur, or unsharp mask (sharpen).
    """
    if p <= 0:
        return x
    B, C, H, W = x.shape
    device = x.device

    apply_mask = torch.rand(B, device=device) < p
    if not apply_mask.any():
        return x

    # Decide which of the applying samples get blur vs sharpen
    apply_indices = torch.nonzero(apply_mask).flatten()
    n_apply = apply_indices.numel()
    n_blur = int(round(n_apply * prob_blur))
    # Shuffle
    perm = apply_indices[torch.randperm(n_apply, device=device)]
    blur_indices = perm[:n_blur]
    sharp_indices = perm[n_blur:]

    out = x.clone()

    # --- Blur branch ---
    if blur_indices.numel() > 0:
        k_choices = torch.tensor(blur_ks_choices, device=device)
        # kernel per sample
        k_idx = torch.randint(0, len(k_choices), (blur_indices.numel(),), device=device)
        k_sizes = k_choices[k_idx]
        sigmas = torch.empty(blur_indices.numel(), 2, device=device).uniform_(*blur_sigma_range)
        for ks in k_choices.unique():
            sel = (k_sizes == ks)
            if not sel.any():
                continue
            idx_sel = blur_indices[sel]
            x_sub = x[idx_sel]
            sigma_sub = sigmas[sel]
            blur_op = K.filters.GaussianBlur2d((int(ks.item()), int(ks.item())), sigma_sub)
            out[idx_sel] = blur_op(x_sub)

    # --- Sharpen (unsharp mask) branch ---
    if sharp_indices.numel() > 0:
        sigmas = torch.empty(sharp_indices.numel(), 2, device=device).uniform_(*usm_sigma_range)
        amounts = torch.empty(sharp_indices.numel(), 1, 1, 1, device=device).uniform_(*usm_amount_range)
        blur_op = K.filters.GaussianBlur2d((usm_ksize, usm_ksize), sigmas)
        base = x[sharp_indices]
        blurred = blur_op(base)
        sharpened = base + amounts * (base - blurred)
        if clamp:
            sharpened = sharpened.clamp(0,1)
        out[sharp_indices] = sharpened

    return out

class ComposeAugmentations:
    def __init__(self, ops: List[Callable[[Tensor], Tensor]]):
        self.ops = ops
    @torch.no_grad()
    def __call__(self, x: Tensor) -> Tensor:
        for op in self.ops:
            x = op(x)
        return x

def build_default_augmentations() -> ComposeAugmentations:
    return ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),
        lambda x: random_gaussian_noise(x, p=0.3),
        lambda x: random_contrast(x, p=0.3),
        lambda x: random_gamma(x, p=0.3),
    ])


In [ ]:
batch_size = dataset_info.batch_size

In [ ]:

train_transformed_loader = dataset_dict['train_loader_transformed']
batch_size = dataset_info.batch_size

In [ ]:
 #test images
fig, axs = plt.subplots(nrows=4, ncols=1, figsize=(10, 6))

axs[0].imshow(torchvision.utils.make_grid(next(iter(train_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(next(iter(train_transformed_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[2].imshow(torchvision.utils.make_grid(next(iter(val_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[3].imshow(torchvision.utils.make_grid(next(iter(test_loader_transformed))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Training')
axs[1].set_title('Training Transformed')
axs[2].set_title('Validation')
axs[3].set_title('Test')

for ax in axs.flat:
    ax.axis('off')

In [ ]:
from model.classifier import Classifier, MyProgressBar
import pytorch_lightning as pl
import os

model_path = f'../model/{dataset}_rot_only.pth'

model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),  # Output: 32x28x28
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 32x14x14
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # Output: 64x14x14
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((7, 7)),                            # Output: 64x7x7
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)


# Check if model is already trained
if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model.load_state_dict(torch.load(model_path))
else:
    print(f"Training model and will save to {model_path}")
    lightning_model = Classifier(model, optimizer_class =  torch.optim.AdamW, optimizer_params = {"lr": 1e-3})
    progress_bar = MyProgressBar()
    trainer = pl.Trainer(
        accelerator="cuda",
        max_epochs=10,
        precision="16-mixed",
        callbacks=[progress_bar],
    )
    # Train the model
    trainer.fit(lightning_model, train_loader,val_loader)
    # Test the model
    #trainer.test(lightning_model, test_loader_transformed)
    # Save model
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


In [ ]:

from equiadapt.images.canonicalization.discrete_group import OptimizedGroupEquivariantImageCanonicalization

In [ ]:
from equiadapt.images.canonicalization_networks import (
    ConvNetwork,
    CustomEquivariantNetwork,
    ESCNNEquivariantNetwork,
    ESCNNSteerableNetwork,
    ESCNNWRNEquivariantNetwork,
    ResNet18Network,
    WideResNet50Network,
    WideResNet101Network,
)

In [ ]:

from omegaconf import OmegaConf
canonicalization_network = ConvNetwork(
    in_shape=(1, 28, 28),
    out_vector_size=128,num_layers=3,kernel_size=3,out_channels=32
)
#remove the dropout layers if any
canonicalization_network.final_fc = nn.Sequential(*[layer for layer in canonicalization_network.final_fc if not isinstance(layer, nn.Dropout1d)])

dict_config = OmegaConf.create({
    "group_type": "rotation",
    "num_rotations":8,
    "artifact_err_wt":0,
    "learn_ref_vec" :False,
    "input_crop_ratio": 1.0,
    "resize_shape": 28,
    "beta": 1.0,
    "gradient_trick":None
})

canonicalizer = OptimizedGroupEquivariantImageCanonicalization(canonicalization_network,
dict_config,(1, 28, 28)).cuda()

In [ ]:
canonicalization_network

In [ ]:
model.cuda().eval()

In [ ]:
def test():
    with torch.no_grad():
        avg_acc = 0
        avg_loss = 0
        for batch in tqdm.tqdm(val_loader_transformed):
            x, y = batch
            x = x.cuda()
            x_canonicalized = canonicalizer(x)
            logits = model(x_canonicalized)
            output = logits.argmax(dim=-1)
            loss = nn.CrossEntropyLoss()(logits, y.cuda())
            acc = output.eq(y.cuda()).sum().item()
            avg_acc += acc
            avg_loss += loss.item()
        avg_acc /= len(val_loader.dataset)
        print(f'Accuracy on the test set after canonicalization: {avg_acc}.')
        print(f'Loss on the test set after canonicalization: {avg_loss/len(val_loader)}.')
import tqdm
opt = torch.optim.Adam(canonicalizer.parameters(), lr=1e-3)
for i in range(10):
    pbar = tqdm.tqdm(train_loader)
    for batch in pbar:
        opt.zero_grad()
        x, y = batch
        x = x.cuda()
        #augment with gaussian blur
        #blur_kernel_size = 3
        #x = random_blur_or_sharpen(x, p=0.5, blur_ks_choices=(3,), prob_blur=1.0)

        x_canonicalized = canonicalizer(x)

        loss =  torch.zeros(1, device='cuda')
        logits = model(x_canonicalized)
        task_loss = torch.nn.functional.cross_entropy(logits, y.cuda())
        loss =  loss + task_loss
        prior_loss = canonicalizer.get_prior_regularization_loss()
        loss += prior_loss*100.0
        specific_loss = canonicalizer.get_optimization_specific_loss()
        loss += specific_loss*1e-4
        loss.backward()
        opt.step()
        #log in obar
        pbar.set_description(f"Loss: {loss.item():.4f}, Prior Loss: {prior_loss.item():.4f}, Specific Loss: {specific_loss.item():.4f}, Task Loss: {task_loss.item():.4f}")
    test()



In [ ]:
canonicalizer.eval()

In [ ]:
canonicalizer.eval().cuda()

In [ ]:
#validate on testset
import torchvision.transforms.functional
import kornia
#validate on testset
#plot some samples
with torch.no_grad():
    # 1. Load a batch
    data, target = next(iter(val_loader))
    data, target = data.cuda(), target.cuda()

    #roate by 1/17 of 2pi
    data = torchvision.transforms.functional.rotate(data, angle=45,interpolation=torchvision.transforms.InterpolationMode.BILINEAR)
    #roate by 90 degrees
    #data = torch.rot90(data, k=1, dims=[2, 3])
    #use kornia
    #data = kornia.geometry.transform.rotate(data, angle=torch.tensor(17*3,device=data.device))

    # 2. Apply spatial transformer
    transformed = canonicalizer(data)



    # 3. Get predictions and correctness mask
    preds = model(transformed).argmax(dim=1)
    correct = preds.eq(target)

    # 4. Define frame colors
    green = torch.tensor((143, 194, 62))[:, None] / 255.
    red = torch.tensor((194, 80, 62))[:, None] / 255.

    # 5. Draw colored frames on transformed images
    imgs = transformed.repeat(1, 3, 1, 1)  # ensure 3 channels
    for img, is_correct in zip(imgs, correct):
        color = green if is_correct else red
        img[:, 0, :] = color
        img[:, -1, :] = color
        img[:, :, 0] = color
        img[:, :, -1] = color

    # 6. Plot original vs. framed outputs
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))
    axs[0].imshow(torchvision.utils.make_grid(data.cpu(), nrow=8).permute(1, 2, 0), cmap='gray')
    axs[0].set_title('Original')
    axs[0].axis('off')

    axs[1].imshow(torchvision.utils.make_grid(imgs.cpu(), nrow=8).permute(1, 2, 0))
    axs[1].set_title('Spatial Transformer (framed)')
    axs[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
 #validate on testset
import torchvision.transforms.functional

#validate on testset
#plot some samples
with torch.no_grad():
    # 1. Load a batch
    data, target = next(iter(val_loader))
    data, target = data.cuda(), target.cuda()
    #roate by 1/17 of 2pi
    data = torchvision.transforms.functional.rotate(data, angle=90,interpolation=torchvision.transforms.InterpolationMode.BILINEAR)


    # 2. Apply spatial transformer
    transformed = canonicalizer(data)



    # 3. Get predictions and correctness mask
    preds = model(transformed).argmax(dim=1)
    correct = preds.eq(target)

    # 4. Define frame colors
    green = torch.tensor((143, 194, 62))[:, None] / 255.
    red = torch.tensor((194, 80, 62))[:, None] / 255.

    # 5. Draw colored frames on transformed images
    imgs = transformed.repeat(1, 3, 1, 1)  # ensure 3 channels
    for img, is_correct in zip(imgs, correct):
        color = green if is_correct else red
        img[:, 0, :] = color
        img[:, -1, :] = color
        img[:, :, 0] = color
        img[:, :, -1] = color

    # 6. Plot original vs. framed outputs
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))
    axs[0].imshow(torchvision.utils.make_grid(data.cpu(), nrow=8).permute(1, 2, 0), cmap='gray')
    axs[0].set_title('Original')
    axs[0].axis('off')

    axs[1].imshow(torchvision.utils.make_grid(imgs.cpu(), nrow=8).permute(1, 2, 0))
    axs[1].set_title('Spatial Transformer (framed)')
    axs[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
#take a single images and roate it by steps of 45 degrees then evalute the socre using the learned model from equiadapt. Then in a second experiemtns blur angles 0,90... and deblur 45,135 using unsharp masking and show the results again.


In [ ]:
with torch.no_grad():
    # 1. Load a batch
    data, target = next(iter(val_loader))
    data, target = data.cuda(), target.cuda()
    #now rotate by 45 degrees multiple times




In [ ]:

import torch
import torch.nn as nn
import pytorch_lightning as pl

class ImageClassifierPipeline(pl.LightningModule):
    def __init__(
        self,
        model,
        canon,
        *,
        canonicalization_type: str,
        task_weight: float = 1.0,
        prior_weight: float = 100.0,
        group_contrast_weight: float = 1e-4,
        prediction_lr: float = 1e-3,
        canonicalization_lr: float = 1e-3,
        image_shape=(1, 28, 28),
        num_classes: int = 10,
        detect_nan: bool = True,
            learn_main: bool = False,
    ):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss()
        self.prediction_network = model
        self.canonicalizer = canon
        self.canonicalization_type = canonicalization_type
        self.task_weight = task_weight
        self.prior_weight = prior_weight
        self.group_contrast_weight = group_contrast_weight
        self.prediction_lr = prediction_lr
        self.canonicalization_lr = canonicalization_lr
        self.image_shape = image_shape
        self.num_classes = num_classes
        self.detect_nan = detect_nan
        self.learn_main = learn_main
        self.save_hyperparameters(
            {
                "canonicalization_type": canonicalization_type,
                "task_weight": task_weight,
                "prior_weight": prior_weight,
                "group_contrast_weight": group_contrast_weight,
                "prediction_lr": prediction_lr,
                "canonicalization_lr": canonicalization_lr,
                "image_shape": image_shape,
                "num_classes": num_classes,
            }
        )

    def _check_finite(self, name: str, tensor: torch.Tensor):
        if self.detect_nan and not torch.isfinite(tensor).all():
            raise RuntimeError(f"{name} is NaN/Inf")

    def _check_vectors(self):
        info = getattr(self.canonicalizer, "canonicalization_info_dict", None)
        if not info:
            return
        if "vector_out" in info:
            self._check_finite("vector_out", info["vector_out"])
            _ = info["vector_out"].norm(p=2)
        if "vector_out_dummy" in info:
            self._check_finite("vector_out_dummy", info["vector_out_dummy"])
            _ = info["vector_out_dummy"].norm(p=2)

    def forward(self, x):
        x_canon = self.canonicalizer(x)
        return self.prediction_network(x_canon)

    def training_step(self, batch, batch_idx):
        x, y = batch
        assert x.shape[1:] == self.image_shape
        x_canon = self.canonicalizer(x)
        self._check_vectors()

        total_loss = torch.zeros(1, device=x.device)

        # Task loss
        logits = self.prediction_network(x_canon)
        task_loss = self.loss_fn(logits, y)
        self._check_finite("task_loss", task_loss)
        total_loss = total_loss + self.task_weight * task_loss

        # Group contrast loss
        group_contrast_loss = None
        if "opt" in self.canonicalization_type and self.group_contrast_weight:
            group_contrast_loss = self.canonicalizer.get_optimization_specific_loss()
            self._check_finite("group_contrast_loss", group_contrast_loss)
            total_loss = total_loss + group_contrast_loss * self.group_contrast_weight

        # Prior loss
        prior_loss = None
        identity_metric = None
        if self.prior_weight:
            prior_loss = self.canonicalizer.get_prior_regularization_loss()
            self._check_finite("prior_loss", prior_loss)
            total_loss = total_loss + prior_loss * self.prior_weight
            identity_metric = self.canonicalizer.get_identity_metric()

        self._check_finite("total_loss", total_loss)

        preds = logits.argmax(dim=-1)
        acc = (preds == y).float().mean()

        # Progress bar only: final loss & acc
        self.log("train/loss", total_loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train/acc", acc, prog_bar=True, on_step=True, on_epoch=True)

        # Additional metrics (no prog bar)
        self.log("train/task_loss", task_loss, prog_bar=False, on_step=True, on_epoch=True)
        if group_contrast_loss is not None:
            self.log("train/optimization_specific_loss", group_contrast_loss, prog_bar=False, on_step=True, on_epoch=True)
        if prior_loss is not None:
            self.log("train/prior_loss", prior_loss, prog_bar=False, on_step=True, on_epoch=True)
        if identity_metric is not None:
            self.log("train/identity_metric", identity_metric, prog_bar=False, on_step=True, on_epoch=True)

        return {"loss": total_loss, "acc": acc}

    def validation_step(self, batch, batch_idx):
        x, y = batch
        x_canon = self.canonicalizer(x)
        self._check_vectors()
        logits = self.prediction_network(x_canon)
        task_loss = self.loss_fn(logits, y)
        preds = logits.argmax(dim=-1)
        acc = (preds == y).float().mean()
        self.log("val/acc", acc, prog_bar=True)
        self.log("val/task_loss", task_loss, prog_bar=False)
        if self.prior_weight:
            identity_metric = self.canonicalizer.get_identity_metric()
            self.log("val/identity_metric", identity_metric, prog_bar=False)
        return {"acc": acc}

    def test_step(self, batch, batch_idx):
        x, y = batch
        x_canon = self.canonicalizer(x)
        self._check_vectors()
        logits = self.prediction_network(x_canon)
        preds = logits.argmax(dim=-1)
        acc = (preds == y).float().mean()
        self.log("test/acc", acc, prog_bar=True)
        if self.prior_weight:
            identity_metric = self.canonicalizer.get_identity_metric()
            self.log("test/identity_metric", identity_metric, prog_bar=False)
        return {"acc": acc}

    def configure_optimizers(self):
        if self.learn_main:
            return torch.optim.AdamW(
                [
                    {"params": [p for p in self.prediction_network.parameters() if p.requires_grad],
                     "lr": self.prediction_lr},
                    {"params": [p for p in self.canonicalizer.parameters() if p.requires_grad],
                     "lr": self.canonicalization_lr},
                ]
            )



        return torch.optim.AdamW(
            [
                {"params": [p for p in self.canonicalizer.parameters() if p.requires_grad],
                 "lr": self.canonicalization_lr},
            ]
        )

In [ ]:
import confidence.supervised.ml.equiadapt
import importlib
importlib.reload(confidence.supervised.ml.equiadapt)

from confidence.supervised.ml.equiadapt import EquiCanonicalizationConfidence

In [ ]:
canon3 = ConvNetwork(
    in_shape=(1, 28, 28),
    out_vector_size=128,num_layers=3,kernel_size=3,out_channels=32
)

In [ ]:
canon3 = torch.nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),  # Output: 32x28x28
    nn.BatchNorm2d(32),                                    # Batch Normalization
    nn.GELU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 32x14x14
    nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),  # Output: 64x14x14
    nn.BatchNorm2d(32),                                    # Batch Normalization
    nn.GELU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 64x7x7
    nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),  # Output: 64x7x7
    nn.BatchNorm2d(32),                                    # Batch Normalization
    nn.GELU(),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.GELU(),
    nn.Linear(128, 128),

)

In [ ]:
from utils.sampling_strategy import GaussianSamplingStrategyLatent, TransformLatentSamplingStrategy
import importlib
import utils.sampling_strategy
import utils.sampling
importlib.reload(utils.sampling)
from utils.sampling import BatchNegativeSampler

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq,),transform_true_function
    = augment_true,augment_function=build_default_augmentations(),
    decision_strategy =None, number_of_negatives=3,return_params=True
)

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint

trainer_kwargs = {
    "max_epochs": 20,
    "callbacks": [MyProgressBar(),ModelCheckpoint(monitor="val_loss",mode="min",save_top_k=1)],
}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Dict, Any, Tuple, Literal
from confidence.unsupervised.unsupervised_base import MLConfidenceBase


class EquiCanonicalizationConfidence(MLConfidenceBase):
    def __init__(
        self,
        canonicalization_network: nn.Module,
        main_model: nn.Module,
        loss_main_model: nn.Module = torch.nn.CrossEntropyLoss(),
        vector_dim: Optional[int] = None,
        beta: float = 1.0,
        task_loss_multiplier: float = 1.0,
        prior_loss_multiplier: float = 1.0,
        diversity_loss_multiplier: float = 1.0,
        reg_loss_multiplier: float = 0.0,
        similarity_metric: Literal["cosine", "dot", "mse", "logit"] = "cosine",
        learn_main_model: bool = False,
        learn_ref_vec: bool = False,
        trainer_kwargs: Optional[Dict[str, Any]] = None,
        dataloader_kwargs: Optional[Dict[str, Any]] = None,
        optimizer_type: Optional[torch.optim.Optimizer] = torch.optim.Adam,
        optimizer_kwargs: Optional[Dict[str, Any]] = None,
        negative_sampling_module: Optional[Any] = None,
        input_transform: Optional[Any] = None,
        gradient_passthrough: Literal["gumbel", "ste"] = "gumbel",
        diversity_loss_type: Literal["gram_abs", "variance", "uniformity"] = "variance",
        use_parameterized_transforms: bool = True,  # NEW: selection method
        gumbel_temperature: float = 1.0,  # Temperature for Gumbel-Softmax
    ):
        super().__init__(
            input_transform=input_transform,
            trainer_kwargs=trainer_kwargs,
            dataloader_kwargs=dataloader_kwargs,
            optimizer_type=optimizer_type,
            optimizer_kwargs=optimizer_kwargs,
            negative_sampling_module=negative_sampling_module,
        )

        self.canon_net = canonicalization_network
        self.main_model = main_model
        self.beta = float(beta)
        self.loss_main_model = loss_main_model
        self.task_loss_multiplier = float(task_loss_multiplier)
        self.prior_loss_multiplier = float(prior_loss_multiplier)
        self.diversity_loss_multiplier = float(diversity_loss_multiplier)
        self.reg_loss_multiplier = float(reg_loss_multiplier)
        self.similarity_metric = similarity_metric
        self.out_vector_size = vector_dim
        self.use_parameterized_transforms = use_parameterized_transforms
        self.gumbel_temperature = gumbel_temperature

        if self.similarity_metric in ("cosine", "dot", "mse"):
            if self.out_vector_size is None:
                raise ValueError("vector_dim must be provided for similarity_metric != 'logit'.")
            self.reference_vector = nn.Parameter(
                torch.randn(1, self.out_vector_size),
                requires_grad=learn_ref_vec,
            )
        else:
            self.reference_vector = nn.Parameter(torch.zeros(1, 1), requires_grad=False)

        self.gradient_passthrough = gradient_passthrough
        self.learn_main_model = learn_main_model
        self.diversity_loss_type = diversity_loss_type

        # Check if negative sampling module returns params
        if negative_sampling_module is not None:
            if not hasattr(negative_sampling_module, 'return_params'):
                raise ValueError("negative_sampling_module must have 'return_params' attribute")
            if use_parameterized_transforms and not negative_sampling_module.return_params:
                raise ValueError("use_parameterized_transforms=True requires negative_sampling_module.return_params=True")

            # Store transform_sequence for applying transforms
            if hasattr(negative_sampling_module.strategy, 'transform_sequence'):
                self.transform_sequence = negative_sampling_module.strategy.transform_sequence
            else:
                raise ValueError("negative_sampling_module.strategy must have 'transform_sequence' attribute")

    def _split_orbit(self, x: torch.Tensor, batch_size: int) -> Tuple[torch.Tensor, int]:
        B_total = x.shape[0]
        assert B_total % batch_size == 0
        K = B_total // batch_size
        new_shape = (K, batch_size) + tuple(x.shape[1:])
        x_reshaped = x.view(*new_shape)
        return x_reshaped.permute(1, 0, *range(2, x_reshaped.dim())), K

    def _compute_scores(self, z: torch.Tensor) -> torch.Tensor:
        if self.similarity_metric == "cosine":
            return F.cosine_similarity(z, self.reference_vector, dim=-1, eps=1e-8)
        elif self.similarity_metric == "dot":
            return (z * self.reference_vector).sum(dim=-1)
        elif self.similarity_metric == "mse":
            return -((z - self.reference_vector) ** 2).mean(dim=-1)
        elif self.similarity_metric == "logit":
            return z if z.dim() == 1 else z[..., 0]
        raise ValueError(f"Unknown similarity_metric {self.similarity_metric}")

    def _canonicalizer_pass(self, x: torch.Tensor, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self.canon_net(x)
        if z.dim() == 1:
            z = z.unsqueeze(-1)
        scores = self._compute_scores(z)
        scores_reshaped, _ = self._split_orbit(scores, batch_size)
        return z, scores_reshaped

    def _diversity_loss(self, Z: torch.Tensor) -> torch.Tensor:
        # Z: [B, K, D] already grouped
        B, K, D = Z.shape
        if K < 2 or self.diversity_loss_multiplier == 0.0:
            return Z.new_zeros(())

        if self.diversity_loss_type == "gram_abs":
            G = Z @ Z.transpose(-1, -2)  # B,K,K
            mask = 1.0 - torch.eye(K, device=Z.device, dtype=Z.dtype).unsqueeze(0)
            return (G.abs() * mask).mean()

        Zc = Z - Z.mean(dim=1, keepdim=True)

        if self.diversity_loss_type == "variance":
            var = Zc.var(dim=1, unbiased=False)  # B,D
            std = torch.sqrt(var + 1e-6)
            return F.relu(1.0 - std).mean()

        if self.diversity_loss_type == "uniformity":
            Zn = F.normalize(Zc, dim=-1)
            alpha = 2.0
            d2 = torch.cdist(Zn, Zn, p=2).pow(2)  # B,K,K
            mask = ~torch.eye(K, dtype=torch.bool, device=Z.device)
            exp_term = torch.exp(-alpha * d2[:, mask].view(B, -1))
            return torch.log(exp_term.mean(dim=1) + 1e-6).mean()

        raise ValueError(f"Unknown diversity_loss_type {self.diversity_loss_type}")

    def select_group_act(self, Q: torch.Tensor) -> torch.Tensor:
        """Discrete selection for image-based approach (not used in parameterized)."""
        if self.gradient_passthrough == "gumbel":
            return F.gumbel_softmax(Q, tau=self.gumbel_temperature, hard=True, dim=-1)
        # STE
        one_hot = torch.one_hot(Q.argmax(dim=-1), num_classes=Q.shape[1]).to(Q.dtype)
        soft = F.softmax(self.beta * Q, dim=-1)
        return one_hot - soft.detach() + soft

    def _select_transform_params(
        self,
        S: torch.Tensor,
        T_shaped: torch.Tensor
    ) -> torch.Tensor:
        """
        Select transform parameters using gradient estimators.

        Args:
            S: Scores [batch_size, K]
            T_shaped: Transform parameters [batch_size, K, ...] (can be multi-dimensional)

        Returns:
            Selected transform parameters [batch_size, ...]
        """
        batch_size, K = S.shape
        # T_shaped can have shape [batch_size, K, transform_dim] or [batch_size, K, ...]
        # We need to handle arbitrary trailing dimensions

        if self.gradient_passthrough == "gumbel":
            # Gumbel-Softmax: samples from categorical distribution with reparameterization
            # Returns soft weights that are differentiable
            selection_weights = F.gumbel_softmax(
                self.beta * S,
                tau=self.gumbel_temperature,
                hard=False,  # Keep soft for continuous gradients
                dim=-1
            )  # [batch_size, K]

            # Weighted sum of transform parameters
            # Need to broadcast selection_weights to match T_shaped dimensions
            # T_shaped: [batch_size, K, ...], selection_weights: [batch_size, K]
            # Reshape to [batch_size, K, 1, 1, ...] to broadcast correctly
            num_extra_dims = len(T_shaped.shape) - 2
            for _ in range(num_extra_dims):
                selection_weights = selection_weights.unsqueeze(-1)

            T_selected = (selection_weights * T_shaped).sum(dim=1)  # [batch_size, ...]

        elif self.gradient_passthrough == "ste":
            # Straight-Through Estimator:
            # Forward pass uses hard selection, backward pass uses soft gradients
            soft_weights = F.softmax(self.beta * S, dim=-1)  # [batch_size, K]

            # Hard selection (one-hot)
            hard_selection = torch.zeros_like(soft_weights)
            hard_indices = S.argmax(dim=-1)
            hard_selection.scatter_(1, hard_indices.unsqueeze(1), 1.0)

            # STE: forward uses hard, backward uses soft
            selection_weights = hard_selection - soft_weights.detach() + soft_weights

            # Select transform parameters
            # Broadcast to match T_shaped dimensions
            num_extra_dims = len(T_shaped.shape) - 2
            for _ in range(num_extra_dims):
                selection_weights = selection_weights.unsqueeze(-1)

            T_selected = (selection_weights * T_shaped).sum(dim=1)  # [batch_size, ...]

        else:
            raise ValueError(f"Unknown gradient_passthrough: {self.gradient_passthrough}")

        return T_selected

    def _compute_losses_discrete(self, x: torch.Tensor, y: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Original discrete selection method."""
        in_dist = y >= 0.0
        batch_size = in_dist.sum().item()
        assert batch_size > 0 and x.shape[0] % batch_size == 0

        z, S = self._canonicalizer_pass(x, batch_size)
        one_hot_selected = self.select_group_act(S)

        x_shaped = self._split_orbit(x, batch_size)[0]
        x_selected = (one_hot_selected.reshape(batch_size, -1, *([1] * (x.dim() - 1))) * x_shaped).sum(dim=1)

        main_res = self.main_model(x_selected)
        y_in = y[in_dist]
        main_loss = self.loss_main_model(main_res, y_in)

        dataset_prior = torch.zeros(batch_size, dtype=torch.long, device=x.device)
        prior_loss = F.cross_entropy(S, dataset_prior)

        Z = self._split_orbit(z, batch_size)[0]
        diversity_loss = self._diversity_loss(Z)

        reg_loss = (z ** 2).mean() if self.reg_loss_multiplier > 0.0 else z.new_zeros(())

        return {
            "task_loss": main_loss,
            "prior_loss": prior_loss,
            "diversity_loss": diversity_loss,
            "reg_loss": reg_loss,
            "S_mean": S.mean(),
        }

    def _compute_losses_parameterized(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        T_params: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        """New parameterized transform method using transform_sequence with gradient estimators."""
        in_dist = y >= 0.0
        batch_size = in_dist.sum().item()
        assert batch_size > 0 and x.shape[0] % batch_size == 0

        # Get positive samples
        x_pos = x[in_dist]
        y_pos = y[in_dist]

        # Get all orbit samples for computing scores
        z, S = self._canonicalizer_pass(x, batch_size)

        # Split transform params into orbits [batch_size, K, transform_dim]
        T_shaped, K_orbits = self._split_orbit(T_params, batch_size)

        # Select transform parameters using Gumbel-Softmax or STE
        # This is the key difference: we select PARAMETERS, not images
        T_selected = self._select_transform_params(S, T_shaped)  # [batch_size, transform_dim]

        # Apply the selected transform to positive images
        # Gradients flow through T_selected back to the canonicalization network
        x_canonicalized = self.transform_sequence.application_method(x_pos, T_selected)

        # Task loss
        main_res = self.main_model(x_canonicalized)
        main_loss = self.loss_main_model(main_res, y_pos)

        # Prior loss: encourage selecting the identity (first element)
        dataset_prior = torch.zeros(batch_size, dtype=torch.long, device=x.device)
        prior_loss = F.cross_entropy(S, dataset_prior)

        # Diversity loss
        Z = self._split_orbit(z, batch_size)[0]
        diversity_loss = self._diversity_loss(Z)

        # Regularization
        reg_loss = (z ** 2).mean() if self.reg_loss_multiplier > 0.0 else z.new_zeros(())

        return {
            "task_loss": main_loss,
            "prior_loss": prior_loss,
            "diversity_loss": diversity_loss,
            "reg_loss": reg_loss,
            "S_mean": S.mean(),
        }

    def _training_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        if self.use_parameterized_transforms:
            # Expect batch to be tuple (modified_batch, T_params) from BatchNegativeSampler
            if isinstance(batch, tuple) and len(batch) == 2:
                actual_batch, T_params = batch
                x, y = actual_batch
            else:
                raise ValueError(
                    "use_parameterized_transforms=True requires BatchNegativeSampler "
                    "with return_params=True, which should return (batch, T_params)"
                )
            losses = self._compute_losses_parameterized(x, y, T_params)
        else:
            x, y = batch
            losses = self._compute_losses_discrete(x, y)

        loss = (
            self.task_loss_multiplier * losses["task_loss"]
            + self.prior_loss_multiplier * losses["prior_loss"]
            + self.diversity_loss_multiplier * losses["diversity_loss"]
            + self.reg_loss_multiplier * losses["reg_loss"]
        )

        self.log_dict(
            {
                "train_task": losses["task_loss"],
                "train_prior": losses["prior_loss"],
                "train_diversity": losses["diversity_loss"],
                "train_reg": losses["reg_loss"],
                "train_S_mean": losses["S_mean"],
            },
            on_step=True,
            on_epoch=True,
        )
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def _validation_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        if self.use_parameterized_transforms:
            if isinstance(batch, tuple) and len(batch) == 2:
                actual_batch, T_params = batch
                x, y = actual_batch
            else:
                raise ValueError(
                    "use_parameterized_transforms=True requires BatchNegativeSampler "
                    "with return_params=True"
                )
            with torch.no_grad():
                losses = self._compute_losses_parameterized(x, y, T_params)
        else:
            x, y = batch
            with torch.no_grad():
                losses = self._compute_losses_discrete(x, y)

        loss = (
            self.task_loss_multiplier * losses["task_loss"]
            + self.prior_loss_multiplier * losses["prior_loss"]
            + self.diversity_loss_multiplier * losses["diversity_loss"]
            + self.reg_loss_multiplier * losses["reg_loss"]
        )

        self.log_dict(
            {
                "val_task": losses["task_loss"],
                "val_prior": losses["prior_loss"],
                "val_diversity": losses["diversity_loss"],
                "val_reg": losses["reg_loss"],
                "val_S_mean": losses["S_mean"],
            },
            on_step=False,
            on_epoch=True,
        )
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def _forward(self, x: torch.Tensor, y: Optional[torch.Tensor] = None):
        in_dist = y >= 0.0 if y is not None else torch.ones(x.shape[0], dtype=torch.bool, device=x.device)
        batch_size = in_dist.sum().item() if in_dist.any() else x.shape[0]
        _, sim = self._canonicalizer_pass(x, batch_size=batch_size)
        return sim

    def configure_optimizers(self):
        if self.learn_main_model:
            return self.optimizer_type(self.parameters(), **self.optimizer_kwargs)
        params = list(self.canon_net.parameters())
        if self.reference_vector is not None:
            params.append(self.reference_vector)
        return self.optimizer_type(params, **self.optimizer_kwargs)

In [ ]:
conf = EquiCanonicalizationConfidence(canonicalization_network=canon3,main_model=model,vector_dim=128,negative_sampling_module=negative_sampling_module,prior_loss_multiplier=0.01,diversity_loss_multiplier=0.001,reg_loss_multiplier=0.1,task_loss_multiplier=1,trainer_kwargs=trainer_kwargs,similarity_metric="cosine",learn_main_model=False,diversity_loss_type="variance",learn_ref_vec=False,gradient_passthrough="gumbel",beta=1,use_parameterized_transforms=True).to(device)

In [ ]:
conf.fit(train_loader, val_data =val_loader)
#restore best
#conf = EquiCanonicalizationConfidence.load_from_checkpoint(conf.trainer.checkpoint_callback.best_model_path,canonicalization_network=canon3,main_model=model,vector_dim=128,#negative_sampling_module=negative_sampling_module,prior_loss_multiplier=0.0,diversity_loss_multiplier=0.0,trainer_kwargs=trainer_kwargs,similarity_metric=conf.similarity_metric).to#(device)


In [ ]:
conf.to(device)

In [ ]:
import search.shgo

di = search.shgo.SHGO(selection_method="topk", local_max_steps=10,initial_samples=100,local_runs=1)

In [ ]:
model.cuda().eval()

In [ ]:
pbar = tqdm.tqdm(test_loader_transformed)
total_acc = 0
total_count = 0
with torch.no_grad():
    for data, target in pbar:
        data, target = data.cuda(), target.cuda()
        out = model(data)
        preds = out.argmax(dim=-1)
        acc = preds.eq(target).sum().item()
        total_acc += acc
        total_count += data.shape[0]
        pbar.set_postfix({"accuracy": total_acc / total_count})


In [ ]:
from utils.transformation_problem import TransformationProblem


def evaluate_conf(model, di, problem, test_loader_transformed, max_batch_override=None):
    if max_batch_override is not None:
        problem.max_batch_size = max_batch_override
    with torch.no_grad():
        test_acc_sim = 0
        counter = 0
        pbar = tqdm.tqdm(test_loader_transformed, desc="Evaluating confidence", unit="batch")
        for data, target in pbar:
            data, target = data.cuda(), target.cuda()
            with torch.enable_grad():
                res = di.optimize(problem, data.cuda(), y=target.cuda())
                res = list(res)
            torch.cuda.empty_cache()
            res[0] = res[0].detach()  # Detach the result to avoid gradients
            res[1] = res[1].detach()  # Detach the result to avoid gradients
            # Apply the transformation and get predictions
            x_transformed2 = problem.transform(data.cuda(), res[0])
            logits = model(x_transformed2)
            output = logits.argmax(dim=-1)
            test_acc_sim += output.eq(target).sum().cpu().detach().item()
            counter += output.shape[0]
            #update progress bar
            pbar.set_postfix({"accuracy": test_acc_sim / counter})

    test_acc_sim /= counter
    return test_acc_sim


import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence

conf_mod_nn_pytorch = SinglePassConfidence(model, EnergyConfidence())
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model, di, problem_nn_pytorch, test_loader_transformed, max_batch_override=128)

In [ ]:
conf.cuda().eval()

In [ ]:
problem_energy_pred_conf = TransformationProblem(conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model, di, problem_energy_pred_conf, test_loader_transformed,max_batch_override=1000)

In [ ]:
#TODO if i really want to test against augmented i should imeplemtthe batch negative sampler for the classifier to do the same kinds of augmentations.

In [ ]:
#TODO also implement this in a dataset for efficiency. Also test kornia bicubic interpolation for affine transform maybe it makes distingishing more difficult

In [ ]:
#train an augmented model
model_path_augmented = "../experiment_notebooks/mnist_cnn_augmented.pth"
model_augmented = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),  # Output: 32x28x28
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 32x14x14
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # Output: 64x14x14
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 64x7x7
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

if os.path.exists(model_path_augmented) and False:
    print(f"Loading augmented model from {model_path_augmented}")
    model_augmented.load_state_dict(torch.load(model_path_augmented))
else:
    print(f"Training augmented model and will save to {model_path_augmented}")
    lightning_model_augmented = Classifier(model_augmented, optimizer_class =  torch.optim.AdamW, optimizer_params = {"lr": 1e-3})
    progress_bar = MyProgressBar()
    checkpoint_callback = ModelCheckpoint(monitor="val_acc", mode="max", save_top_k=1)
    trainer = pl.Trainer(
        accelerator="cuda",
        max_epochs=50,
        precision="16-mixed",
        callbacks=[progress_bar, checkpoint_callback],
    )
    # Train the model
    trainer.fit(lightning_model_augmented, train_dataloaders=train_transformed_loader, val_dataloaders=val_loader_transformed)

    #restore best
    lightning_model_augmented = Classifier.load_from_checkpoint(checkpoint_callback.best_model_path, model=model_augmented, optimizer_class =  torch.optim.AdamW, optimizer_params = {"lr": 1e-3})
    model_augmented = lightning_model_augmented.model
    model_augmented.eval()

    # Save the trained model
    torch.save(model_augmented.state_dict(), model_path_augmented)

model_augmented.cuda().eval()



In [ ]:
pbar = tqdm.tqdm(val_loader, desc="Evaluating confidence", unit="batch")
print()
total_acc = 0
total_count = 0
with torch.no_grad():
    for data, target in pbar:
        data, target = data.cuda(), target.cuda()
        out = model(data)
        preds = out.argmax(dim=-1)
        acc = preds.eq(target).sum().item()
        total_acc += acc
        total_count += data.shape[0]
        pbar.set_postfix({"accuracy": total_acc / total_count})

In [ ]:
single_augmented_conf = SinglePassConfidence(model_augmented, EnergyConfidence())
problem_nn_pytorch_augmented = TransformationProblem(single_augmented_conf, transform_seq, consolidate_method="consolidate_simple")

In [ ]:
#test the augmented model
evaluate_conf(model_augmented, di, problem_nn_pytorch_augmented, test_loader_transformed, max_batch_override=128)